In [82]:
import numpy
import illustris_python as il

basePath='/public/share/chenhouzun/TNG50-1/output/'
snapNum = 99

In [66]:
def Filter(basePath, snapNum, h=0.6774, 
           log_mstar_bounds=None, 
           log_mbh_bounds=None, 
           sfr_bounds=None, 
           log_ssfr_bounds=None, 
           log_mhalo_bounds=None):

    
    # 1. 动态确定需要加载的 Halo 字段
    halo_fields = ['GroupFirstSub']
    if log_mhalo_bounds is not None:
        halo_fields.append('GroupMass')
        
    halos = il.groupcat.loadHalos(basePath, snapNum, fields=halo_fields)
    
    # 统一将返回结果包装为字典
    if not isinstance(halos, dict):
        halos = {halo_fields[0]: halos}
    
    valid_halo_mask = halos['GroupFirstSub'] != -1
    subhalo_ids = halos['GroupFirstSub'][valid_halo_mask]
    
    final_mask = np.ones(len(subhalo_ids), dtype=bool)
    
    # --- 主晕层级筛选 ---
    if log_mhalo_bounds is not None:
        mhalo_sim = halos['GroupMass'][valid_halo_mask]
        min_sim = (10**log_mhalo_bounds[0]) * h / 1e10
        max_sim = (10**log_mhalo_bounds[1]) * h / 1e10
        final_mask &= (mhalo_sim >= min_sim) & (mhalo_sim <= max_sim)

    # 2. 动态确定需要加载的 Subhalo 字段
    subhalo_fields = set()
    if log_mstar_bounds or log_ssfr_bounds:
        subhalo_fields.add('SubhaloMassType')
    if log_mbh_bounds:
        subhalo_fields.add('SubhaloBHMass')
    if sfr_bounds or log_ssfr_bounds:
        subhalo_fields.add('SubhaloSFR')
        
    if not subhalo_fields:
        return subhalo_ids[final_mask]
        
    subhalo_fields_list = list(subhalo_fields)
    subhalos = il.groupcat.loadSubhalos(basePath, snapNum, fields=subhalo_fields_list)
    
    # 统一将返回结果包装为字典
    if not isinstance(subhalos, dict):
        subhalos = {subhalo_fields_list[0]: subhalos}
    
    # --- 子晕层级筛选 ---
    if log_mstar_bounds is not None:
        mstar_sim = subhalos['SubhaloMassType'][subhalo_ids, 4]
        min_sim = (10**log_mstar_bounds[0]) * h / 1e10
        max_sim = (10**log_mstar_bounds[1]) * h / 1e10
        final_mask &= (mstar_sim >= min_sim) & (mstar_sim <= max_sim)
        
    if log_mbh_bounds is not None:
        mbh_sim = subhalos['SubhaloBHMass'][subhalo_ids]
        min_sim = (10**log_mbh_bounds[0]) * h / 1e10
        max_sim = (10**log_mbh_bounds[1]) * h / 1e10
        final_mask &= (mbh_sim >= min_sim) & (mbh_sim <= max_sim)
        
    if sfr_bounds is not None:
        sfr = subhalos['SubhaloSFR'][subhalo_ids]
        final_mask &= (sfr >= sfr_bounds[0]) & (sfr <= sfr_bounds[1])
        
    if log_ssfr_bounds is not None:
        mstar_physical = subhalos['SubhaloMassType'][subhalo_ids, 4] * 1e10 / h
        sfr = subhalos['SubhaloSFR'][subhalo_ids]
        
        nonzero_mask = (mstar_physical > 0) & (sfr > 0)
        ssfr = np.zeros_like(sfr)
        ssfr[nonzero_mask] = sfr[nonzero_mask] / mstar_physical[nonzero_mask]
        
        log_ssfr = np.full_like(ssfr, -np.inf)
        log_ssfr[nonzero_mask] = np.log10(ssfr[nonzero_mask])
        
        final_mask &= (log_ssfr >= log_ssfr_bounds[0]) & (log_ssfr <= log_ssfr_bounds[1])

    return subhalo_ids[final_mask]

In [67]:
import time                # 从操作系统路径模块中导入 isfile 函数，用于检查特定路径下是否存在某文件（用于实现缓存机制）
from os.path import isfile # 导入 h5py 库。HDF5 是一种专门用于存储超大规模数值数据的文件格式，在天体物理模拟中极其常见
import h5py                # 导入 NumPy 库并简写为 np。NumPy 是 Python 科学计算的核心库，提供高性能的多维数组（ndarray）对象及矩阵运算
import numpy as np         
from numba import jit      # 从 numba 库导入 jit 装饰器。Numba 是一个即时编译器（Just-In-Time），能将 Python 代码编译为机器码执行，极大提升循环密集型代码的速度

In [68]:
# =========================================================================
# 通用工具与物理计算模块
# =========================================================================

def periodic_displacement(pos1, pos2, boxSize):
    """计算考虑周期性边界条件的最短位移向量 dx"""
    return (pos1 - pos2 + 0.5 * boxSize) % boxSize - 0.5 * boxSize

def periodic_distance(pos1, pos2, boxSize):
    """处理宇宙学周期性边界条件的距离计算"""
    dx = periodic_displacement(pos1, pos2, boxSize)
    return np.linalg.norm(dx, axis=-1)

def calculate_virial_velocity(m_vir, r_vir):
    """计算主晕的维里速度 V_vir (单位: km/s)"""
    G = 43.009  # Gadget/Arepo 内部单位制引力常数
    return np.sqrt(G * m_vir / r_vir)

In [69]:
# =========================================================================
# 模块一：空间过滤
# =========================================================================

def _get_target_geometry(basePath, snapNum, subhalo_id, rmin_fac, rmax_fac):
    """获取目标子晕和母晕的空间几何与物理基准信息"""
    subhalo_data = il.groupcat.loadSingle(basePath, snapNum, subhaloID=subhalo_id)
    center_pos = subhalo_data['SubhaloPos']
    center_vel = subhalo_data['SubhaloVel']
    parent_group_id = subhalo_data['SubhaloGrNr']
    
    group_data = il.groupcat.loadSingle(basePath, snapNum, haloID=parent_group_id)
    rvir = group_data['Group_R_Crit200']
    m_vir = group_data['Group_M_Crit200']
    r200c = group_data['Group_R_Crit200']
    
    r_min = rmin_fac * rvir
    r_max = rmax_fac * rvir
    
    return center_pos, center_vel, parent_group_id, rvir, m_vir, r200c, r_min, r_max

def _get_snapshot_metadata(basePath, snapNum):
    """定位快照文件路径并获取全局元数据"""
    snap_path = il.snapshot.snapPath(basePath, snapNum)
    if os.path.isdir(snap_path):
        file_prefix = os.path.join(snap_path, f"snap_{snapNum:03d}")
    else:
        file_prefix = snap_path.split('.0.hdf5')[0].split('.hdf5')[0]

    first_chunk_file = f"{file_prefix}.0.hdf5"
    if not os.path.exists(first_chunk_file):
        first_chunk_file = f"{file_prefix}.hdf5" 
        
    with h5py.File(first_chunk_file, 'r') as f:
        header = f['Header'].attrs
        num_files = header['NumFilesPerSnapshot']
        boxSize = header['BoxSize']
        total_gas = header['NumPart_Total'][0] + (header['NumPart_Total_HighWord'][0] << 32)
        
    return file_prefix, first_chunk_file, num_files, boxSize, total_gas

def _compute_spatial_mask(file_prefix, first_chunk_file, num_files, boxSize, total_gas, center_pos, r_min, r_max):
    """逐块加载核心与内存安全距离过滤"""
    global_valid_indices = []
    valid_distances = []
    current_global_offset = 0 
    
    for i in range(num_files):
        chunk_file = f"{file_prefix}.{i}.hdf5" if num_files > 1 else first_chunk_file
        with h5py.File(chunk_file, 'r') as f:
            num_gas_this_chunk = f['Header'].attrs['NumPart_ThisFile'][0]
            if num_gas_this_chunk == 0:
                current_global_offset += num_gas_this_chunk
                continue
                
            pos_chunk = f['PartType0/Coordinates'][()]
            dx = periodic_displacement(pos_chunk, center_pos, boxSize)
            
            bbox_mask = (np.abs(dx[:, 0]) <= r_max) & \
                        (np.abs(dx[:, 1]) <= r_max) & \
                        (np.abs(dx[:, 2]) <= r_max)
                        
            bbox_valid_inds = np.where(bbox_mask)[0]
            
            if len(bbox_valid_inds) > 0:
                dx_bbox = dx[bbox_valid_inds]
                dist_candidates = np.linalg.norm(dx_bbox, axis=-1)
                
                exact_mask = (dist_candidates >= r_min) & (dist_candidates <= r_max)
                local_valid_inds = bbox_valid_inds[exact_mask]
                
                if len(local_valid_inds) > 0:
                    global_valid_inds = local_valid_inds + current_global_offset
                    global_valid_indices.append(global_valid_inds)
                    valid_distances.append(dist_candidates[exact_mask])
            
            current_global_offset += num_gas_this_chunk

    if len(global_valid_indices) > 0:
        final_global_indices = np.concatenate(global_valid_indices)
        final_distances = np.concatenate(valid_distances)
    else:
        final_global_indices, final_distances = np.array([], dtype=np.int64), np.array([], dtype=np.float32)

    spatial_mask = np.zeros(total_gas, dtype=bool)
    spatial_mask[final_global_indices] = True
    
    return spatial_mask, final_distances, final_global_indices

def _remove_satellite_subhalos(basePath, snapNum, subhalo_id, center_pos, boxSize, r_max, buffer_kpc, spatial_mask):

    # 1. 批量加载存在于主 Catalog 的字段，避免上万次循环读取
    fields = ['SubhaloPos', 'SubhaloLenType']
    sub_data = il.groupcat.loadSubhalos(basePath, snapNum, fields=fields)
    
    all_sub_pos = sub_data['SubhaloPos']
    all_len_type = sub_data['SubhaloLenType']
    
    # 2. 向量化计算所有子晕到中心的距离，进行空间初筛
    sub_dists = periodic_distance(all_sub_pos, center_pos, boxSize)
    nearby_sub_ids = np.where(sub_dists <= r_max + buffer_kpc)[0]
    
    removed_sub_count = 0
    removed_gas_count = 0
    
    # 3. 仅对空间上真正邻近的子晕进行深入处理（此时数量已经极少，循环毫无压力）
    for sid in nearby_sub_ids:
        if sid == subhalo_id:
            continue
            
        # PartType0 为气体
        len_gas = all_len_type[sid, 0]
        
        # 只有当该子晕真的含有气体时，才去昂贵地读取 Offset 文件
        if len_gas > 0:
            try:
                # 安全调用官方 API 获取独立存储的 Offset
                offsets = il.snapshot.getSnapOffsets(basePath, snapNum, sid, "Subhalo")
                offset_gas = offsets['offsetType'][0]
                
                overlap_count = np.sum(spatial_mask[offset_gas : offset_gas + len_gas])
                if overlap_count > 0:
                    spatial_mask[offset_gas : offset_gas + len_gas] = False # 原地修改掩膜
                    removed_sub_count += 1
                    removed_gas_count += overlap_count
            except Exception as e:
                # 容错：防止个别损坏的子晕中断整个大循环
                print(f"[警告]跳过子晕 {sid} 的剔除，读取 Offset 失败: {e}")
                
    return removed_sub_count, removed_gas_count

def stream_spatial_mask(sP, basePath, snapNum, subhalo_id, rmin_fac, rmax_fac, buffer_kpc):
    """主控调度函数：生成冷流空间掩膜，并精确剔除视野内所有的卫星子晕。"""
    print(f"正在处理 Subhalo ID: {subhalo_id} 的空间掩膜 ---")
    start_time = time.time()

    center_pos, center_vel, parent_group_id, rvir, m_vir, r200c, r_min, r_max = _get_target_geometry(
        basePath, snapNum, subhalo_id, rmin_fac, rmax_fac
    )
    
    file_prefix, first_chunk_file, num_files, boxSize, total_gas = _get_snapshot_metadata(
        basePath, snapNum
    )
    print(f"全局气体总数: {total_gas}。开始执行分块流式空间过滤...")

    spatial_mask, final_distances, final_global_indices = _compute_spatial_mask(
        file_prefix, first_chunk_file, num_files, boxSize, total_gas, center_pos, r_min, r_max
    )
    
    print("开始执行卫星星系去除程序...")
    removed_sub_count, removed_gas_count = _remove_satellite_subhalos(
        basePath, snapNum, subhalo_id, center_pos, boxSize, r_max, buffer_kpc, spatial_mask
    )

    num_valid_cells = np.sum(spatial_mask)
    print(f"子晕剔除完毕：共发现了 {removed_sub_count} 个穿透的卫星星系，移除了 {removed_gas_count} 个受污染网格。")
    print(f"保留了 {num_valid_cells} 个纯净冷流候选网格。总耗时: {time.time()-start_time:.2f}s")
    
    subhalo_info = {
        "subhalo_id": subhalo_id,
        "center_pos": center_pos,
        "center_vel": center_vel,
        "parent_group_id": parent_group_id,
        "rvir": rvir,
        "r200c": r200c,
        "m_vir": m_vir,
        "distance": final_distances[spatial_mask[final_global_indices]]
    }
    
    return spatial_mask, subhalo_info

In [70]:
# =========================================================================
# 模块二：物理条件过滤
# =========================================================================

def generate_physical_mask(sP, basePath, snapNum, spatial_mask, subhalo_info):
    """在空间掩膜基础上，应用温度、密度、动力学条件的复合筛选。"""
    print(f"正在进行物理条件过滤...")
    
    candidate_indices = np.where(spatial_mask)[0]
    num_candidates = len(candidate_indices)
    
    if num_candidates == 0:
        return np.zeros_like(spatial_mask, dtype=bool)

    nH = sP.snapshotSubset("gas", "nh")[candidate_indices]
    temp = sP.snapshotSubset("gas", "temp_sfcold")[candidate_indices]
    sfr = sP.snapshotSubset("gas", "sfr")[candidate_indices]
    pos = sP.snapshotSubset("gas", "pos")[candidate_indices]
    vel = sP.snapshotSubset("gas", "vel")[candidate_indices] 
    
    sf_mask_local = sfr > 0.0
    temp[sf_mask_local] = 1e3
    
    mask_temp_local = (temp >= 5e3) & (temp <= 2.5e5)
    mask_dens_local = (nH >= 1e-4) & (nH <= 1.0)
    
    center_vel = subhalo_info['center_vel']
    boxSize = sP.boxSize
    
    dx = periodic_displacement(pos, subhalo_info['center_pos'], boxSize)
    dv = vel - center_vel
    
    dist = np.linalg.norm(dx, axis=-1)
    dist = np.clip(dist, 1e-5, None) 
    
    v_rad_local = np.sum(dv * dx, axis=1) / dist
    v_tot_local = np.linalg.norm(dv, axis=-1)
    v_tot_local = np.clip(v_tot_local, 1e-5, None)
    
    v_vir = calculate_virial_velocity(subhalo_info['m_vir'], subhalo_info['rvir'])
    
    mask_vrad_limit_local = v_rad_local < (-0.2 * v_vir)
    mask_vratio_local = (np.abs(v_rad_local) / v_tot_local) > 0.8
    
    valid_local_mask = (mask_temp_local & mask_dens_local & mask_vrad_limit_local & mask_vratio_local)
    
    print(f"物理条件筛选完毕：保留 {np.sum(valid_local_mask)} 个符合冷流的网格。(V_vir = {v_vir:.2f} km/s)")
    
    final_valid_mask = np.zeros_like(spatial_mask, dtype=bool)
    final_valid_mask[candidate_indices] = valid_local_mask
    return final_valid_mask

In [71]:
# =========================================================================
# 模块三：连通域标记算法
# =========================================================================

def load_stream_voronoi_mesh(sP):
    """读取整个模拟快照 Snapshot 全局的 Voronoi 网格邻接数据。"""
    filename = sP.derivPath + "voronoi/mesh_%02d.hdf5" % sP.snap
    print("正在加载全局 Voronoi 拓扑图...")
    start_t = time.time()
    
    with h5py.File(filename, "r") as f:
        num_ngb = f["num_ngb"][()]
        offset_ngb = f["offset_ngb"][()]
        ngb_inds = f["ngb_inds"][()]
        
    print(f"拓扑图加载完成，耗时: {time.time()-start_t:.2f}s")
    return num_ngb, ngb_inds, offset_ngb

@jit(nopython=True, nogil=True)
def _contiguousStreamCells(num_ngb, offset_ngb, ngb_inds, valid_mask, valid_indices, identity):
    """基于布尔掩膜的稀疏连通域标记算法"""
    n_valid = valid_indices.size
    count = 0 

    for k in range(n_valid):
        i = valid_indices[k]
        for j in range(num_ngb[i]):
            ngb_index = offset_ngb[i] + j
            cell_index_j = ngb_inds[ngb_index]

            if cell_index_j == -1 or not valid_mask[cell_index_j]:
                continue

            if identity[cell_index_j] >= 0:
                identity[i] = identity[cell_index_j]
                break 

        if identity[i] < 0:
            identity[i] = count
            count += 1 

    converged = False
    for _niter in range(1000):
        changes_count = 0
        for k in range(n_valid):
            i = valid_indices[k]
            for j in range(num_ngb[i]):
                ngb_index = offset_ngb[i] + j
                cell_index_j = ngb_inds[ngb_index]

                if cell_index_j == -1 or not valid_mask[cell_index_j]:
                    continue

                if identity[cell_index_j] >= 0 and identity[cell_index_j] < identity[i]:
                    identity[i] = identity[cell_index_j]
                    changes_count += 1 

        if changes_count == 0:
            converged = True
            break
            
    if not converged:
        print("警告: 连通域弛豫过程在 1000 次迭代后未完全收敛！")

    c = np.zeros(count, dtype=np.int32) - 1
    for k in range(n_valid):
        i = valid_indices[k]
        if identity[i] >= 0:
            c[identity[i]] = 1 

    new_count = 0
    for idx in range(c.size):
        if c[idx] > 0:
            c[idx] = new_count 
            new_count += 1

    for k in range(n_valid):
        i = valid_indices[k]
        if identity[i] >= 0:
            identity[i] = c[identity[i]]

    return new_count 

def identify_stream_components(sP, final_valid_mask):
    """接收输出掩膜，返回打好标签的身份数组。"""
    print(f"正在执行冷流拓扑连通域重构...")
    start_time = time.time()
    
    valid_indices = np.where(final_valid_mask)[0]
    n_valid = len(valid_indices)
    
    if n_valid == 0:
        print("未发现任何符合条件的冷流网格，跳过拓扑重构。")
        return np.full_like(final_valid_mask, -1, dtype=np.int32), 0
        
    print(f"进入 JIT 编译核心：将对 {n_valid} 个有效网格进行图遍历...")
    
    num_ngb, ngb_inds, offset_ngb = load_stream_voronoi_mesh(sP)
    identity = np.full(final_valid_mask.shape, -1, dtype=np.int32)
    
    jit_start = time.time()
    num_components = _contiguousStreamCells(num_ngb, offset_ngb, ngb_inds, final_valid_mask, valid_indices, identity)
    
    print(f"拓扑重构完成：共识别出 {num_components} 个独立的冷流分支/团块。")
    print(f"JIT 图遍历耗时: {time.time()-jit_start:.2f}s。总耗时: {time.time()-start_time:.2f}s")
    
    return identity, num_components

In [72]:
# =========================================================================
# 模块四：形态学与高级过滤 (I/O 性能重构)
# =========================================================================

def load_surviving_gas_properties(sP, identity):
    """统一提取存活网格的物理量，避免离散切片读取 (I/O 优化)"""
    valid_mask = identity >= 0
    valid_indices = np.where(valid_mask)[0]
    
    if len(valid_indices) == 0:
        return None, valid_indices
        
    preloaded_data = {
        "pos": sP.snapshotSubset("gas", "pos")[valid_indices],
        "vol": sP.snapshotSubset("gas", "volume")[valid_indices],
        "mass": sP.snapshotSubset("gas", "mass")[valid_indices]
    }
    return preloaded_data, valid_indices


def filter_streams_by_virial_shell(sP, subhalo_info, identity, count, buffer_ratio, preloaded_data=None, valid_indices=None):
    """通过与 R200c 薄球壳的交集，过滤掉未穿过维里边界的孤立冷气体团块"""
    if count == 0:
        return identity, 0
        
    print(f"开始执行维里边界薄球壳交集筛选...")
    r200c = subhalo_info['r200c']
    
    R_inner = r200c * (1.0 - buffer_ratio)
    R_outer = r200c * (1.0 + buffer_ratio)
    print(f"定义 R200c 薄球壳范围: [{R_inner:.2f}, {R_outer:.2f}] kpc/h (R200c = {r200c:.2f})")
    
    if preloaded_data is None or valid_indices is None:
        valid_mask = identity >= 0
        valid_indices = np.where(valid_mask)[0]
        pos = sP.snapshotSubset("gas", "pos")[valid_indices]
    else:
        pos = preloaded_data["pos"]
        
    local_identities = identity[valid_indices]
    center_pos = subhalo_info['center_pos']
    dist = periodic_distance(pos, center_pos, sP.boxSize)
    
    in_shell_mask = (dist >= R_inner) & (dist <= R_outer)
    surviving_ids = np.unique(local_identities[in_shell_mask])
    
    print(f"初筛的 {count} 个团块中，有 {len(surviving_ids)} 个结构与 R200c 球壳相交。")
    
    c = np.zeros(count, dtype=np.int32) - 1
    if len(surviving_ids) > 0:
        c[surviving_ids] = np.arange(len(surviving_ids))
        
    identity[valid_indices] = c[local_identities]
    new_count = len(surviving_ids)
    
    print(f"交集筛选完成：剩余 {new_count} 个冷气体团块。")
    return identity, new_count


def filter_streams_morphology(sP, subhalo_info, identity, count, preloaded_data=None, valid_indices=None):
    """基于主成分分析 (PCA) 与形态学的冷流高级过滤器 (严格物理计算版)。"""
    if count == 0:
        return identity, 0
        
    print(f"开始执行 PCA 形态学与严格形状张量分析...")
    
    if preloaded_data is None or valid_indices is None:
        valid_mask = identity >= 0
        valid_indices = np.where(valid_mask)[0]
        pos = sP.snapshotSubset("gas", "pos")[valid_indices]
        vol = sP.snapshotSubset("gas", "volume")[valid_indices]
        mass = sP.snapshotSubset("gas", "mass")[valid_indices]
    else:
        pos = preloaded_data["pos"]
        vol = preloaded_data["vol"]
        mass = preloaded_data["mass"]
        
    local_identities = identity[valid_indices]
    center_pos = subhalo_info['center_pos']
    boxSize = sP.boxSize
    r200c = subhalo_info['r200c']
    
    clump_volumes = np.bincount(local_identities, weights=vol, minlength=count)
    max_vol = np.max(clump_volumes)
    vol_threshold = 0.05 * max_vol
    print(f"最大候选体物理体积: {max_vol:.2e} [kpc/h]^3，相对 5% 截断阈值: {vol_threshold:.2e} [kpc/h]^3")

    surviving_ids = []

    for i in range(count):
        if clump_volumes[i] < vol_threshold:
            continue
            
        clump_mask = (local_identities == i)
        clump_pos = pos[clump_mask]
        clump_mass = mass[clump_mask] 
        
        dist = periodic_distance(clump_pos, center_pos, boxSize)
        if np.max(dist) < 1.5 * r200c:
            continue
            
        dx = periodic_displacement(clump_pos, center_pos, boxSize)
        clump_cen_masswt = np.average(dx, axis=0, weights=clump_mass)
        dx_centered = dx - clump_cen_masswt
        
        cov_matrix = np.cov(dx_centered, rowvar=False, aweights=clump_mass)
        
        try:
            eigenvalues, _ = np.linalg.eigh(cov_matrix)
            # 严格数值保护：强制截断浮点误差造成的微小负数，防止 np.sqrt 爆 NaN
            eigenvalues = np.clip(eigenvalues, 0.0, None)
            eigenvalues = np.sort(eigenvalues)[::-1]
        except np.linalg.LinAlgError:
            print(f"提示: 团块 {i} 出现病态协方差矩阵 (LinAlgError)，跳过该结构。")
            continue
            
        if eigenvalues[1] <= 1e-10 or eigenvalues[2] <= 1e-10:
            continue
            
        a = np.sqrt(eigenvalues[0])
        b = np.sqrt(eigenvalues[1])
        c = np.sqrt(eigenvalues[2])
        
        axis_ratio_ab = a / b
        axis_ratio_ac = a / c
        
        if axis_ratio_ab > 5.0 and axis_ratio_ac > 10.0:
            surviving_ids.append(i)

    print(f"形态学筛选完成：保留了 {len(surviving_ids)} 个冷流结构。")

    c_map = np.zeros(count, dtype=np.int32) - 1
    if len(surviving_ids) > 0:
        c_map[surviving_ids] = np.arange(len(surviving_ids))
        
    identity[valid_indices] = c_map[local_identities]
    new_count = len(surviving_ids)
    
    return identity, new_count

In [73]:
# =========================================================================
# 模块五：物理属性提取与持久化
# =========================================================================

def _load_and_sort_property(sP, field_name, valid_indices, sort_idx, fallback_field=None):
    data = sP.snapshotSubset("gas", field_name)
    if data is None and fallback_field is not None:
        data = sP.snapshotSubset("gas", fallback_field)
    
    if data is None:
        return None
        
    return data[valid_indices][sort_idx]


def extract_stream_properties(sP, basePath, snapNum, subhalo_info, identity, count):
    """提取被验证为冷流的网格集群宏观物理属性，并保存到 HDF5 文件。"""
    saveFilename = sP.derivPath + f"voronoi/cold_streams_snap{snapNum:03d}_sh{subhalo_info['subhalo_id']}.hdf5"
    
    if count == 0:
        print(f"提取阶段：未检测到任何冷流，写入空占位文件。")
        with h5py.File(saveFilename, "w") as f:
            f["objects/count"] = 0
            f["props/dummy"] = 0
        return {"count": 0}, {"dummy": 0}

    print(f"开始提取 {count} 条冷流的物理与几何属性...")
    start_time = time.time()
    
    valid_indices = np.where(identity >= 0)[0]
    local_identities = identity[valid_indices]
    
    lengths = np.bincount(local_identities, minlength=count)
    
    sort_idx = np.argsort(local_identities, kind="mergesort")
    cell_inds = valid_indices[sort_idx]
    
    offsets = np.zeros(count, dtype="int32")
    offsets[1:] = np.cumsum(lengths)[:-1]

    print("正在按连续内存块重载所需物理量...")
    vol   = _load_and_sort_property(sP, "volume", valid_indices, sort_idx)
    mass  = _load_and_sort_property(sP, "mass", valid_indices, sort_idx)
    dens  = _load_and_sort_property(sP, "nh", valid_indices, sort_idx)
    temp  = _load_and_sort_property(sP, "temp_sfcold", valid_indices, sort_idx)
    pos   = _load_and_sort_property(sP, "pos", valid_indices, sort_idx)
    vel   = _load_and_sort_property(sP, "vel", valid_indices, sort_idx)
    metal = _load_and_sort_property(sP, "z_solar", valid_indices, sort_idx)
    
    vrel = vel - subhalo_info['center_vel']
    dx = periodic_displacement(pos, subhalo_info['center_pos'], sP.boxSize)
    dist_all = np.linalg.norm(dx, axis=-1)
    dist_all = np.clip(dist_all, 1e-5, None)
    vrad = np.sum(vrel * dx, axis=1) / dist_all
    
    props = {
        "count": np.array([count], dtype="int32"),
        "vol": np.zeros(count, dtype="float32"),
        "mass": np.zeros(count, dtype="float32"),
        "dens_mean": np.zeros(count, dtype="float32"),
        "temp_mean": np.zeros(count, dtype="float32"),
        "metal_mean": np.zeros(count, dtype="float32"),
        "vrad_mean": np.zeros(count, dtype="float32"),
        "cen": np.zeros((count, 3), dtype="float32"),         
        "cen_masswt": np.zeros((count, 3), dtype="float32"),  
        "vrel_masswt": np.zeros((count, 3), dtype="float32"), 
        "vrel_denswt": np.zeros((count, 3), dtype="float32"),
    }

    for i in range(count):
        loc = slice(offsets[i], offsets[i] + lengths[i])
        
        props["vol"][i] = vol[loc].sum()
        props["mass"][i] = mass[loc].sum()
        props["dens_mean"][i] = dens[loc].mean()
        props["temp_mean"][i] = temp[loc].mean()
        props["metal_mean"][i] = metal[loc].mean()
        props["vrad_mean"][i] = vrad[loc].mean()

        props["cen"][i, :] = np.average(pos[loc, :], axis=0)
        props["cen_masswt"][i, :] = np.average(pos[loc, :], axis=0, weights=mass[loc])
        props["vrel_masswt"][i, :] = np.average(vrel[loc, :], axis=0, weights=mass[loc])
        props["vrel_denswt"][i, :] = np.average(vrel[loc, :], axis=0, weights=dens[loc])

    props["distance"] = periodic_distance(props["cen_masswt"], subhalo_info['center_pos'], sP.boxSize)

    objects = {
        "count": count,
        "lengths": lengths,
        "offsets": offsets,
        "cell_inds": cell_inds,
    }

    with h5py.File(saveFilename, "w") as f:
        for key in objects: f["objects/%s" % key] = objects[key]
        for key in props: f["props/%s" % key] = props[key]

    print(f"数据已保存至: [{saveFilename.split(sP.derivPath)[-1]}]")
    print(f"模块五属性提取完成，耗时: {time.time()-start_time:.2f}s")
    
    return objects, props

In [74]:
# =========================================================================
# 终极主控流水线
# =========================================================================

def analyze_cold_streams_pipeline(sP, basePath, snapNum, subhalo_id, 
                                  rmin_fac, rmax_fac, buffer_kpc, buffer_ratio,
                                  output_dir):
    """输入模拟快照与目标子晕，输出并保存该子晕周围高度纯净的冷流网格簇及物理属性。"""
    os.makedirs(output_dir, exist_ok=True)
    saveFilename = os.path.join(output_dir, f"cold_streams_snap{snapNum:03d}_sh{subhalo_id}.hdf5")
    
    # 修复：补充了 os.path 
    if os.path.isfile(saveFilename):
        print(f"检测到已存在的冷流数据缓存: {saveFilename}，跳过计算。")
        objects, props = {}, {}
        with h5py.File(saveFilename, "r") as f:
            for key in f["objects"]: objects[key] = f["objects"][key][()]
            for key in f["props"]: props[key] = f["props"][key][()]
        return objects, props

    print(f"\n{'='*60}")
    print(f"开始进行冷流提取 - 快照: {snapNum}, 目标 Subhalo: {subhalo_id}")
    print(f"{'='*60}")
    pipeline_start = time.time()

    # [步骤 1] 空间包围盒与卫星星系剔除
    spatial_mask, subhalo_info = stream_spatial_mask(
        sP, basePath, snapNum, subhalo_id, rmin_fac, rmax_fac, buffer_kpc
    )

    # [步骤 2] 热力学与动力学多维物理切断
    final_valid_mask = generate_physical_mask(
        sP, basePath, snapNum, spatial_mask, subhalo_info
    )

    # [步骤 3] Numba 底层 Voronoi 稀疏拓扑连通域重构
    identity, count = identify_stream_components(sP, final_valid_mask)

    # [步骤 4] 内存提取重载 (服务于高级过滤器)
    preloaded_data, valid_indices = load_surviving_gas_properties(sP, identity)

    # [步骤 5] 高级形态学双重验证：R200c 球壳交集 + 严格 PCA 形状张量分析
    identity, count = filter_streams_by_virial_shell(
        sP, subhalo_info, identity, count, 
        buffer_ratio=0.05, preloaded_data=preloaded_data, valid_indices=valid_indices
    )
    
    identity, count = filter_streams_morphology(
        sP, subhalo_info, identity, count,
        preloaded_data=preloaded_data, valid_indices=valid_indices
    )

    del preloaded_data 

    # [步骤 6] 物理属性集群宏观提取与持久化保存
    objects, props = extract_stream_properties(
        sP, basePath, snapNum, subhalo_info, identity, count
    )

    print(f"{'='*60}")
    print(f"冷流识别结束，全局总耗时: {time.time()-pipeline_start:.2f}s。")
    print(f"{'='*60}\n")
    
    return objects, props

In [86]:
class RealSP:
    """
    轻量级的 TNG 数据读取与物理转换代理类
    专门用于对接 Cold Stream Pipeline
    """
    def __init__(self, basePath, snapNum, boxSize, derivPath):
        self.basePath = basePath
        self.snapNum = snapNum
        self.snap = snapNum      # Pipeline 中使用了 sP.snap
        self.boxSize = boxSize
        
        # 确保衍生数据目录（如 voronoi 文件夹）存在
        self.derivPath = derivPath
        os.makedirs(os.path.join(self.derivPath, "voronoi"), exist_ok=True)
        
        # 内存缓存，避免同一次执行中重复读取几十GB的文件
        self._cache = {}

    def snapshotSubset(self, partType, fieldName):
        if partType != "gas":
            return None
            
        # 如果已经读过，直接从内存缓存返回
        if fieldName in self._cache:
            return self._cache[fieldName]

        print(f"[底层 I/O] 正在从快照加载全局大字段: {fieldName} ... (可能需要几十秒到几分钟)")
        
        # 基础坐标与动力学属性
        if fieldName == "pos":
            data = il.snapshot.loadSubset(self.basePath, self.snapNum, 'gas', fields=['Coordinates'])
        elif fieldName == "vel":
            data = il.snapshot.loadSubset(self.basePath, self.snapNum, 'gas', fields=['Velocities'])
        elif fieldName == "mass":
            data = il.snapshot.loadSubset(self.basePath, self.snapNum, 'gas', fields=['Masses'])
            
        # TNG 衍生几何与物理属性计算
        elif fieldName == "volume":
            mass = self.snapshotSubset("gas", "mass")
            rho = il.snapshot.loadSubset(self.basePath, self.snapNum, 'gas', fields=['Density'])
            data = mass / rho
            
        elif fieldName == "nh":
            # 计算氢原子数密度 (cm^-3)
            rho = il.snapshot.loadSubset(self.basePath, self.snapNum, 'gas', fields=['Density'])
            XH = 0.76  # 氢质量分数
            m_p = 1.6726e-24 # 质子质量(克)
            UnitMass_in_g = 1.989e43 / 0.6774 # 10^10 Msun / h
            UnitLength_in_cm = 3.085678e21 / 0.6774 # kpc / h
            rho_cgs = rho * (UnitMass_in_g / UnitLength_in_cm**3)
            data = (rho_cgs * XH) / m_p
            
        elif fieldName == "temp_sfcold":
            # 计算温度 (K)
            ie = il.snapshot.loadSubset(self.basePath, self.snapNum, 'gas', fields=['InternalEnergy'])
            xe = il.snapshot.loadSubset(self.basePath, self.snapNum, 'gas', fields=['ElectronAbundance'])
            gamma = 5.0 / 3.0
            k_B = 1.3806e-16
            m_p = 1.6726e-24
            XH = 0.76
            mu = 4.0 / (1.0 + 3.0 * XH + 4.0 * XH * xe) * m_p
            # 内部能量单位转换：(km/s)^2 -> (cm/s)^2 -> 1e10
            data = (gamma - 1.0) * (ie * 1e10) * mu / k_B 
            
        elif fieldName == "sfr":
            data = il.snapshot.loadSubset(self.basePath, self.snapNum, 'gas', fields=['StarFormationRate'])
            
        elif fieldName == "z_solar":
            # 金属丰度 (GFM_Metallicity 默认是无量纲的质量分数，直接返回即可，如果需要严格转换为 Z_sun 可除以 0.0127)
            data = il.snapshot.loadSubset(self.basePath, self.snapNum, 'gas', fields=['GFM_Metallicity'])
            
        else:
            print(f"[警告] 未知的字段请求: {fieldName}")
            return None
            
        self._cache[fieldName] = data
        return data

    def clear_cache(self):
        """释放内存：如果您在循环分析多个 subhalo，且内存不足，可调用此方法"""
        self._cache.clear()
        import gc
        gc.collect()
        print("[系统] 全局内存缓存已释放。")

In [84]:
subhalo_ids = Filter(basePath, snapNum, h=0.6774, 
              log_mstar_bounds=(11.0, 12.0), 
              log_mbh_bounds=None, 
              sfr_bounds=None, 
              log_ssfr_bounds=(-12.0, -11.0), 
              log_mhalo_bounds=None)

print(f"共提取 {len(subhalo_ids)} 个测试中心星系")

共提取 21 个测试中心星系


In [ ]:
import traceback

# ==========================================
# 1. 初始化路径与模拟基准参数
# ==========================================
basePath = '/public/share/chenhouzun/TNG50-1/output/'  
snapNum = 99
boxSize = 35000.0 
derivPath = "/public/home/zju_visitor/LiuYuanhao/simulation_results/derived/"

print("正在初始化虚拟模拟上下文环境...")
sP = RealSP(basePath=basePath, snapNum=snapNum, boxSize=boxSize, derivPath=derivPath)

# ==========================================
# 2. 循环分析与进度统计
# ==========================================
success_count = 0
fail_count = 0

# 为了测试，这里先用 [:1] 只跑第一个。跑通后可以把 [:1] 去掉跑全部任务。
test_subhalos = subhalo_ids[:1]

for i, shID in enumerate(test_subhalos):
    print(f"\n进度: [{i+1}/{len(test_subhalos)}] 正在分析 Subhalo ID: {shID}")
    
    try:
        objects, props = analyze_cold_streams_pipeline(
            sP, 
            basePath, 
            snapNum, 
            shID, 
            rmin_fac=0.15, 
            rmax_fac=3.0, 
            buffer_kpc=500.0,
            buffer_ratio=0.05,
            output_dir="/public/home/zju_visitor/LiuYuanhao/simulation_results"
        )
        
        # 判断是否识别到了有效的冷流
        if objects['count'] > 0:
            print(f">>> 成功：在子晕 {shID} 周围识别到 {objects['count']} 条冷流。")
        else:
            print(f">>> 提示：子晕 {shID} 周围未发现符合条件的冷流。")
            
        success_count += 1
        
    except Exception as e:
        print(f"错误：在处理子晕 {shID} 时发生异常: {e}")
        traceback.print_exc()  # 打印出具体的错误行号，方便调试
        fail_count += 1
        continue

# ==========================================
# 3. 统计输出
# ==========================================
print(f"\n{'#'*40}")
print(f"所有任务处理完成")
print(f"成功处理: {success_count}")
print(f"失败任务: {fail_count}")
print(f"结果已保存至: /public/home/zju_visitor/LiuYuanhao/simulation_results")
print(f"{'#'*40}")

# 如果您一次性跑了几十个星系，结束时可以手动释放一下全局缓存
# sP.clear_cache()

正在初始化虚拟模拟上下文环境...

进度: [1/1] 正在分析 Subhalo ID: 229933

开始进行冷流提取 - 快照: 99, 目标 Subhalo: 229933
正在处理 Subhalo ID: 229933 的空间掩膜 ---
全局气体总数: 8737803526。开始执行分块流式空间过滤...
开始执行卫星星系去除程序...
子晕剔除完毕：共发现了 41 个穿透的卫星星系，移除了 1467344 个受污染网格。
保留了 53898999 个纯净冷流候选网格。总耗时: 913.46s
正在进行物理条件过滤...
    [底层 I/O] 正在从快照加载全局大字段: nh ... (可能需要几十秒到几分钟)
